# 03 — CNN más profunda, con filtrado (regularización)

Mismo notebook que 02 (señal filtrada, sin balanceo, umbral fijo 0.5), cambiando únicamente la arquitectura: de un bloque convolucional mínimo a los tres bloques (32→64→128 filtros) con BatchNorm, Dropout y regularización L2 del modelo principal. Aísla cuánto aporta la capacidad del modelo frente al preprocesado, que ya se evaluó en el salto 01→02.

## 1. Entorno, semillas y librerías

In [1]:
import sys, os, random, hashlib, json
from pathlib import Path

import numpy as np
import pandas as pd
import ast

import wfdb
from scipy import signal as sp_signal
from scipy.signal import sosfiltfilt, iirnotch, tf2sos

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.regularizers import l2

import matplotlib.pyplot as plt

print(sys.executable)
print("TensorFlow:", tf.__version__)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


C:\Users\Usuario\anaconda3\envs\tfg\python.exe
TensorFlow: 2.16.1


## 2. Configuración general

`NOTEBOOK_NAME` y `PREP_VERSION` son las dos variables que cambian entre notebooks de esta serie: identifican la fila en la tabla comparativa y la carpeta de caché, respectivamente.

In [2]:
NOTEBOOK_NAME = "03_cnn_profunda_con_filtrado"
PREP_VERSION  = "con_filtrado"     # reutiliza el mismo caché que 02, la señal no cambia

DATA_DIR  = Path(r"C:\Users\Usuario\Desktop\TFG\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
CACHE_DIR = DATA_DIR / "cache_v2" / PREP_VERSION
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path("./outputs/models"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR   = Path("./outputs/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_CSV = MODEL_DIR / "model_comparison.csv"

FS = 100
SUPER_CLASSES = ["NORM", "MI", "STTC", "HYP", "CD"]
CLASS_MAP_HUMAN = {
    "NORM": "Normal", "MI": "Infarto", "STTC": "Isquemia/ST-T",
    "HYP": "Hipertrofia", "CD": "Conducción (bloqueos)",
}

BANDPASS_LOW  = 0.5
BANDPASS_HIGH = 40.0
NOTCH_FREQ    = 50.0
NOTCH_Q       = 30.0

STFT_NPERSEG  = 64
STFT_NOVERLAP = 48
SPEC_SHAPE    = (33, 64)

L2_REG = 1e-4   # fuerza de la regularización L2


## 3. Carga de metadatos y etiquetas multi-etiqueta

In [3]:
df  = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
df.columns  = df.columns.str.strip()
scp.columns = scp.columns.str.strip()

scp_super = (scp[scp["diagnostic_class"].notna()][["diagnostic_class"]]
             .rename(columns={"diagnostic_class": "superclass"}))

def extract_superclasses(scp_codes_str):
    """Convierte el string tipo diccionario de scp_codes en la lista de superclases presentes."""
    if pd.isna(scp_codes_str):
        return []
    try:
        d = ast.literal_eval(scp_codes_str)
        if not isinstance(d, dict):
            return []
    except Exception:
        return []
    return list({scp_super.loc[k, "superclass"] for k in d.keys() if k in scp_super.index})

def multilabel_vector(classes):
    """Convierte la lista de superclases en un vector binario de tamaño 5."""
    s = set(classes)
    return np.array([1 if c in s else 0 for c in SUPER_CLASSES], dtype=np.int32)

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].apply(len) > 0].reset_index()
df["target_vec"] = df["superclasses"].apply(multilabel_vector)


## 4. Split por paciente

In [4]:
train_df = df[df["strat_fold"].isin(range(1, 9))].reset_index(drop=True)
val_df   = df[df["strat_fold"] == 9].reset_index(drop=True)
test_df  = df[df["strat_fold"] == 10].reset_index(drop=True)

assert set(train_df.patient_id) & set(val_df.patient_id) == set()
assert set(train_df.patient_id) & set(test_df.patient_id) == set()
assert set(val_df.patient_id)  & set(test_df.patient_id)  == set()

print("Tamaños:", len(train_df), len(val_df), len(test_df))


Tamaños: 17084 2146 2158


## 5. Preprocesado de la señal ECG (limpieza)


In [5]:
def is_valid_signal(x, flat_std_thresh=1e-6):
    """Descarta derivaciones corruptas: con NaN/Inf o completamente planas (fallo de electrodo)."""
    if np.any(np.isnan(x)) or np.any(np.isinf(x)):
        return False
    return np.std(x) >= flat_std_thresh

_BP_SOS    = sp_signal.butter(4, [BANDPASS_LOW / (FS/2), BANDPASS_HIGH / (FS/2)], btype="band", output="sos")
_NOTCH_SOS = tf2sos(*iirnotch(NOTCH_FREQ / (FS/2), NOTCH_Q))

def clean_lead(x):
    """Limpia una derivación (1D): paso-banda + notch + normalización z-score."""
    x = np.asarray(x, dtype=np.float64)
    if not is_valid_signal(x):
        return None
    x = sosfiltfilt(_BP_SOS, x)
    x = sosfiltfilt(_NOTCH_SOS, x)
    mu, sigma = x.mean(), x.std()
    if sigma < 1e-8:
        return None
    return ((x - mu) / sigma).astype(np.float32)

def preprocess_ecg(raw_signal):
    """Limpia las 12 derivaciones de un ECG. Descarta el registro si más de la mitad fallan."""
    cleaned = np.zeros_like(raw_signal, dtype=np.float32)
    n_bad = 0
    for i in range(raw_signal.shape[1]):
        c = clean_lead(raw_signal[:, i])
        cleaned[:, i] = c if c is not None else 0.0
        n_bad += (c is None)
    return None if n_bad > raw_signal.shape[1] // 2 else cleaned


## 6. Extracción de características: espectrograma STFT


In [6]:
def lead_to_spectrogram(lead_signal, fs=FS, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP, out_shape=SPEC_SHAPE):
    """Convierte una derivación 1D en su espectrograma STFT normalizado, con forma fija (F, T)."""
    _, _, Zxx = sp_signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
    mag = np.log1p(np.abs(Zxx))

    F, T = out_shape
    mag = mag[:F, :T]
    if mag.shape != (F, T):
        pad = np.zeros((F, T), dtype=np.float32)
        pad[:mag.shape[0], :mag.shape[1]] = mag
        mag = pad

    mn, mx = mag.min(), mag.max()
    if mx - mn > 1e-8:
        mag = (mag - mn) / (mx - mn)
    return mag.astype(np.float32)

def ecg_to_tensor(signal_2d):
    """signal_2d: (n_muestras, 12) -> tensor (F, T, 12)."""
    specs = [lead_to_spectrogram(signal_2d[:, i]) for i in range(signal_2d.shape[1])]
    return np.stack(specs, axis=-1)


## 7. Generación del dataset con caché en disco

A diferencia de 01, aquí **sí se llama a `preprocess_ecg`** antes de calcular el espectrograma — es la única línea que cambia respecto al notebook anterior en todo este bloque.

In [7]:
def _cache_path(rel_path):
    h = hashlib.md5(rel_path.encode()).hexdigest()
    return CACHE_DIR / f"{h}.npy"

def get_tensor_cached(rel_path):
    cp = _cache_path(rel_path)
    if cp.exists():
        return np.load(cp)
    raw, _ = wfdb.rdsamp(str(DATA_DIR / rel_path))
    clean = preprocess_ecg(raw)
    if clean is None:
        return None
    tensor = ecg_to_tensor(clean)
    np.save(cp, tensor)
    return tensor

def build_split(df_split):
    Xs, Ys = [], []
    for _, row in df_split.iterrows():
        t = get_tensor_cached(row["filename_lr"])
        if t is None:
            continue
        Xs.append(t)
        Ys.append(row["target_vec"])
    return np.stack(Xs).astype(np.float32), np.stack(Ys).astype(np.float32)

X_train, Y_train = build_split(train_df)
X_val,   Y_val   = build_split(val_df)
X_test,  Y_test  = build_split(test_df)

print("Formas:", X_train.shape, X_val.shape, X_test.shape)


Formas: (17084, 33, 64, 12) (2146, 33, 64, 12) (2158, 33, 64, 12)


## 8. Arquitectura: CNN de tres bloques

Tres bloques `Conv2D → BatchNorm → MaxPooling → Dropout` con 32, 64 y 128 filtros respectivamente, regularización L2 en cada convolución y una capa densa de 64 unidades antes de la clasificación. Sigue sin `pos_weight` ni focal loss: eso llega en 04.

In [20]:
def build_model_deep(input_shape, n_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation="sigmoid")(x)

    return models.Model(inputs, outputs)

model = build_model_deep(X_train.shape[1:], len(SUPER_CLASSES))
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc", multi_label=True)])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 33, 64, 12)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 33, 64, 32)          │           3,488 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 33, 64, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 16, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 16, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 16, 32, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 16, 32, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 8, 16, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 8, 16, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 8, 16, 128)          │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 8, 16, 128)          │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 128)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │             325 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 105,317 (411.39 KB)

 Trainable params: 104,869 (409.64 KB)

 Non-trainable params: 448 (1.75 KB)

## 9. Entrenamiento

Sin `pos_weight` ni focal loss: `binary_crossentropy` simple, umbral de decisión fijo en 0.5.

In [21]:
cbs = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=4, min_lr=1e-6),
    callbacks.ModelCheckpoint(str(MODEL_DIR / f"{NOTEBOOK_NAME}.keras"), monitor="val_auc",
                               mode="max", save_best_only=True),
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=60,
    batch_size=32,
    callbacks=cbs,
    verbose=1,
)


Epoch 1/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 56s 94ms/step - auc: 0.7781 - loss: 0.4626 - val_auc: 0.8247 - val_loss: 0.4380 - learning_rate: 0.0010
Epoch 2/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 47s 89ms/step - auc: 0.8384 - loss: 0.4019 - val_auc: 0.8513 - val_loss: 0.4527 - learning_rate: 0.0010
Epoch 3/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 45s 85ms/step - auc: 0.8528 - loss: 0.3847 - val_auc: 0.8613 - val_loss: 0.3918 - learning_rate: 0.0010
Epoch 4/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 86s 92ms/step - auc: 0.8604 - loss: 0.3750 - val_auc: 0.8674 - val_loss: 0.3759 - learning_rate: 0.0010
Epoch 5/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 43s 81ms/step - auc: 0.8667 - loss: 0.3665 - val_auc: 0.8676 - val_loss: 0.3912 - learning_rate: 0.0010
Epoch 6/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 40s 75ms/step - auc: 0.8717 - loss: 0.3607 - val_auc: 0.8686 - val_loss: 0.3705 - learning_rate: 0.0010
Epoch 7/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 41s 76ms/step - auc: 0.8740 - loss: 0.3582 - val_auc: 0.8692 - val_loss: 0.3684 - learning_rate:

## 10. Evaluación sobre el conjunto de test

In [22]:
def compute_metrics(Y_true, Y_pred, Y_prob):
    """Accuracy, Precision, Recall, F1 y MAE por clase, más la fila Macro (media simple)."""
    rows = []
    for i, c in enumerate(SUPER_CLASSES):
        yt, yp, ypr = Y_true[:, i], Y_pred[:, i], Y_prob[:, i]
        tp = np.sum((yt == 1) & (yp == 1))
        tn = np.sum((yt == 0) & (yp == 0))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))
        acc  = (tp + tn) / len(yt)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        mae  = np.mean(np.abs(ypr - yt))
        rows.append([CLASS_MAP_HUMAN[c], acc, prec, rec, f1, mae])

    metrics_df = pd.DataFrame(rows, columns=["Clase", "Accuracy", "Precision", "Recall", "F1", "MAE"])
    macro = metrics_df[["Accuracy", "Precision", "Recall", "F1", "MAE"]].mean()
    metrics_df.loc[len(metrics_df)] = ["Macro"] + macro.tolist()
    return metrics_df

test_probs = model.predict(X_test, verbose=0)
test_pred  = (test_probs >= 0.5).astype(int)

metrics_df = compute_metrics(Y_test, test_pred, test_probs)
metrics_df


,Clase,Accuracy,Precision,Recall,F1,MAE
0,Normal,0.842447,0.770634,0.921080,0.839167,0.187454
1,Infarto,0.823448,0.792388,0.416364,0.545888,0.211895
2,Isquemia/ST-T,0.854958,0.665079,0.804223,0.728063,0.175597
3,Hipertrofia,0.888323,0.565217,0.347328,0.430260,0.155766
4,Conducción (bloqueos),0.862373,0.875472,0.467742,0.609724,0.163442
5,Macro,0.854310,0.733758,0.591347,0.630620,0.178831


## 11. Registro en la tabla comparativa común


In [25]:
macro_row = metrics_df[metrics_df["Clase"] == "Macro"].drop(columns="Clase").iloc[0]
result_row = pd.DataFrame([{
    "Notebook": NOTEBOOK_NAME,
    "Preprocesado": PREP_VERSION,
    "Parametros_entrenables": model.count_params(),
    **macro_row.to_dict(),
}])

if COMPARISON_CSV.exists():
    comparison = pd.read_csv(COMPARISON_CSV)
    comparison = comparison[comparison["Notebook"] != NOTEBOOK_NAME]
    comparison = pd.concat([comparison, result_row], ignore_index=True)
else:
    comparison = result_row

comparison.to_csv(COMPARISON_CSV, index=False)
comparison


,Notebook,Preprocesado,Parametros_entrenables,Accuracy,Precision,Recall,F1,MAE
0,02_cnn_simple_con_filtrado,con_filtrado,3653,0.829472,0.733107,0.387682,0.457371,0.241108
1,03_cnn_profunda_con_filtrado,con_filtrado,105317,0.854310,0.733758,0.591347,0.630620,0.178831
